# City Property Tax Comparisons

This notebook loads Utah property tax rate data from the official Excel file provided by the Utah State Tax Commission, stores it in a SQLite database, and performs analysis and comparisons.

In [29]:
from pathlib import Path
import pandas as pd
import sqlite3

# Setup paths
data_dir = Path("data")
excel_path = data_dir / "2025 Tax Rate by Tax Area - State.xlsx"

In [30]:
# Load tax rate data from official Excel file
print("Loading tax data from official Excel file...")

tax_df = pd.read_excel(excel_path)

print(f"Loaded {len(tax_df)} records")
print(f"\nColumns: {list(tax_df.columns)}")
print(f"\nData types:")
print(tax_df.dtypes)
print(f"\nFirst 10 rows:")
display(tax_df.head(10))

# Basic data quality checks
print(f"\nData quality checks:")
print(f"Total records: {len(tax_df)}")
print(f"Unique counties: {tax_df['County'].nunique()}")
print(f"Unique tax areas: {tax_df['TaxArea'].nunique()}")
print(f"Unique entities: {tax_df['EntityCode'].nunique()}")
print(f"Null CertifiedRate: {tax_df['CertifiedRate'].isna().sum()}")
print(f"Null ProposedRate: {tax_df['ProposedRate'].isna().sum()}")
print(f"Null FinalRate: {tax_df['FinalRate'].isna().sum()}")

Loading tax data from official Excel file...
Loaded 15056 records

Columns: ['TaxYear', 'CodeCounty', 'County', 'TaxArea', 'TaxAreaExtension', 'EntityCode', 'EntityName', 'CertifiedRate', 'ProposedRate', 'FinalRate']

Data types:
TaxYear               int64
CodeCounty            int64
County               object
TaxArea              object
TaxAreaExtension      int64
EntityCode            int64
EntityName           object
CertifiedRate       float64
ProposedRate        float64
FinalRate           float64
dtype: object

First 10 rows:


,TaxYear,CodeCounty,County,TaxArea,TaxAreaExtension,EntityCode,EntityName,CertifiedRate,ProposedRate,FinalRate
0,2025,1,BEAVER,001,0,1010,BEAVER,0.001462,0.001462,0.001462
1,2025,1,BEAVER,001,0,1015,MULTICOUNTY ASSESSING & COLLECTING LEVY,0.000014,0.000014,0.000014
2,2025,1,BEAVER,001,0,1020,COUNTY ASSESSING & COLLECTING LEVY,0.000294,0.000294,0.000294
3,2025,1,BEAVER,001,0,2010,BEAVER COUNTY SCHOOL DISTRICT,0.005190,0.005190,0.005190
4,2025,1,BEAVER,001,0,4010,BEAVER COUNTY SPECIAL SERVICE DISTRICT NO.1,0.000375,0.000375,0.000375
5,2025,1,BEAVER,001,5,1010,BEAVER,0.001462,0.001462,0.001462
6,2025,1,BEAVER,001,5,1015,MULTICOUNTY ASSESSING & COLLECTING LEVY,0.000014,0.000014,0.000014
7,2025,1,BEAVER,001,5,1020,COUNTY ASSESSING & COLLECTING LEVY,0.000294,0.000294,0.000294
8,2025,1,BEAVER,001,5,2010,BEAVER COUNTY SCHOOL DISTRICT,0.005190,0.005190,0.005190
9,2025,1,BEAVER,001,5,4010,BEAVER COUNTY SPECIAL SERVICE DISTRICT NO.1,0.000375,0.000375,0.000375



Data quality checks:
Total records: 15056
Unique counties: 29
Unique tax areas: 797
Unique entities: 138
Null CertifiedRate: 179
Null ProposedRate: 0
Null FinalRate: 0


In [31]:
# Create SQLite database
db_path = data_dir / "property_tax.db"
conn = sqlite3.connect(db_path)

print(f"Creating database at {db_path}")

# Store the Excel data
tax_df.to_sql('tax_rates', conn, if_exists='replace', index=False)
print("Data stored in 'tax_rates' table")

# Verify the data
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM tax_rates")
row_count = cursor.fetchone()[0]
print(f"Total records in database: {row_count}")

# Show table schema
cursor.execute("PRAGMA table_info(tax_rates)")
print("\nTable schema:")
for col in cursor.fetchall():
    print(f"  {col[1]} ({col[2]})")

# Show unique counties
cursor.execute("SELECT DISTINCT County FROM tax_rates ORDER BY County")
counties = [row[0] for row in cursor.fetchall()]
print(f"\nCounties found: {len(counties)}")
print(counties)

conn.close()
print("\nDatabase setup complete")

Creating database at data/property_tax.db
Data stored in 'tax_rates' table
Total records in database: 15056

Table schema:
  TaxYear (INTEGER)
  CodeCounty (INTEGER)
  County (TEXT)
  TaxArea (TEXT)
  TaxAreaExtension (INTEGER)
  EntityCode (INTEGER)
  EntityName (TEXT)
  CertifiedRate (REAL)
  ProposedRate (REAL)
  FinalRate (REAL)

Counties found: 29
['BEAVER', 'BOX ELDER', 'CACHE', 'CARBON', 'DAGGETT', 'DAVIS', 'DUCHESNE', 'EMERY', 'GARFIELD', 'GRAND', 'IRON', 'JUAB', 'KANE', 'MILLARD', 'MORGAN', 'PIUTE', 'RICH', 'SALT LAKE', 'SAN JUAN', 'SANPETE', 'SEVIER', 'SUMMIT', 'TOOELE', 'UINTAH', 'UTAH', 'WASATCH', 'WASHINGTON', 'WAYNE', 'WEBER']

Database setup complete


In [32]:
# Load municipal boundaries data
municipal_csv_path = data_dir / "UtahMunicipalBoundaries_221481185616367289.csv"

print(f"Loading municipal boundaries from {municipal_csv_path}")
municipal_df = pd.read_csv(municipal_csv_path)

print(f"Loaded {len(municipal_df)} municipalities")
print(f"\nColumns: {list(municipal_df.columns)}")
print("\nFirst 5 rows:")
display(municipal_df.head())

Loading municipal boundaries from data/UtahMunicipalBoundaries_221481185616367289.csv
Loaded 261 municipalities

Columns: ['OBJECTID', 'COUNTYNBR', 'NAME', 'COUNTYSEAT', 'SHORTDESC', 'UPDATED', 'FIPS', 'ENTITYNBR', 'SALESTAXID', 'IMSCOLOR', 'MINNAME', 'POPLASTCENSUS', 'POPLASTESTIMATE', 'UGRCODE', 'GNIS', 'GlobalID', 'Shape__Area', 'Shape__Length']

First 5 rows:


,OBJECTID,COUNTYNBR,NAME,COUNTYSEAT,SHORTDESC,UPDATED,FIPS,ENTITYNBR,SALESTAXID,IMSCOLOR,MINNAME,POPLASTCENSUS,POPLASTESTIMATE,UGRCODE,GNIS,GlobalID,Shape__Area,Shape__Length
0,1,3,Newton,0.0,NEWTON,10/20/2020 12:00:00 AM,54550.0,3100.0,47.0,2,,789.0,815.0,NWT,1430705.0,773ccaed-22b4-4237-ba15-17315a94a7d5,4.009109e+06,8948.894827
1,2,12,Eureka,0.0,EUREKA,NaN,24080.0,3010.0,9.0,2,,662.0,657.0,EUR,1437974.0,3ef04b24-1a05-4355-a900-d8c59ff118ae,6.387834e+06,12832.366127
2,3,29,Huntsville,0.0,HUNTSVILLE,5/2/2024 12:00:00 AM,37060.0,3030.0,19.0,4,Huntsville,573.0,593.0,HVL,1428949.0,f0274f4c-4dba-4475-97bb-329012e9c691,1.433332e+07,24584.998375
3,4,27,Springdale,0.0,SPRINGDALE,NaN,71840.0,3100.0,23.0,2,Springdale,514.0,576.0,SPD,1432867.0,9d93a698-af3a-48d3-9a9e-d49b1a39b74c,1.878263e+07,23966.386568
4,5,23,Grantsville,0.0,GRANTSVILLE,2/8/2022 12:00:00 AM,31120.0,3010.0,23.0,2,,12617.0,14417.0,GRA,1428338.0,d6bb102d-36fa-415b-947f-959c60413b85,1.835223e+08,112936.761973


In [33]:
# Transform municipal data
print("Transforming municipal data...")

# Add computed columns
municipal_df['area_sq_miles'] = municipal_df['Shape__Area'] / 2589988  # Convert sq meters to sq miles
municipal_df['normalized_name'] = municipal_df['NAME'].str.upper().str.replace(' CITY', '').str.replace(' TOWN', '').str.strip()

# Handle population - use estimate if available, otherwise census
municipal_df['population'] = municipal_df['POPLASTESTIMATE'].fillna(municipal_df['POPLASTCENSUS'])

# Select relevant columns for database
municipal_clean = municipal_df[[
    'OBJECTID', 'COUNTYNBR', 'NAME', 'SHORTDESC', 'ENTITYNBR', 
    'UGRCODE', 'population', 'Shape__Area', 'area_sq_miles', 'normalized_name'
]].copy()

print(f"Transformed data shape: {municipal_clean.shape}")
print("\nSample of transformed data:")
display(municipal_clean.head(10))

Transforming municipal data...
Transformed data shape: (261, 10)

Sample of transformed data:


,OBJECTID,COUNTYNBR,NAME,SHORTDESC,ENTITYNBR,UGRCODE,population,Shape__Area,area_sq_miles,normalized_name
0,1,3,Newton,NEWTON,3100.0,NWT,815.0,4.009109e+06,1.547926,NEWTON
1,2,12,Eureka,EUREKA,3010.0,EUR,657.0,6.387834e+06,2.466357,EUREKA
2,3,29,Huntsville,HUNTSVILLE,3030.0,HVL,593.0,1.433332e+07,5.534126,HUNTSVILLE
3,4,27,Springdale,SPRINGDALE,3100.0,SPD,576.0,1.878263e+07,7.252013,SPRINGDALE
4,5,23,Grantsville,GRANTSVILLE,3010.0,GRA,14417.0,1.835223e+08,70.858377,GRANTSVILLE
5,6,18,Bluffdale,BLUFFDALE (SL CO),3020.0,BDL,19080.0,4.671062e+07,18.035073,BLUFFDALE
6,7,19,Blanding,BLANDING,3010.0,BLA,3229.0,5.471993e+07,21.127483,BLANDING
7,8,18,Herriman,HERRIMAN TOWN,3035.0,HER,59179.0,1.036384e+08,40.015006,HERRIMAN
8,9,12,Levan,LEVAN,3020.0,LEV,892.0,3.482854e+06,1.344738,LEVAN
9,10,9,Panguitch,PANGUITCH,3070.0,PAN,1785.0,1.345872e+07,5.196440,PANGUITCH


In [34]:
# Load municipal data into SQLite
conn = sqlite3.connect(db_path)

print("Loading municipal data into database...")
municipal_clean.to_sql('municipal_boundaries', conn, if_exists='replace', index=False)

# Verify the data
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM municipal_boundaries")
row_count = cursor.fetchone()[0]
print(f"Total municipal records in database: {row_count}")

# Show table schema
cursor.execute("PRAGMA table_info(municipal_boundaries)")
print("\nTable schema:")
for col in cursor.fetchall():
    print(f"  {col[1]} ({col[2]})")

conn.close()
print("\nMunicipal data loaded successfully")

Loading municipal data into database...
Total municipal records in database: 261

Table schema:
  OBJECTID (INTEGER)
  COUNTYNBR (INTEGER)
  NAME (TEXT)
  SHORTDESC (TEXT)
  ENTITYNBR (REAL)
  UGRCODE (TEXT)
  population (REAL)
  Shape__Area (REAL)
  area_sq_miles (REAL)
  normalized_name (TEXT)

Municipal data loaded successfully


In [35]:
# Add indexes for performance
conn = sqlite3.connect(db_path)

print("Adding indexes for performance...")

cursor = conn.cursor()

# Index on entity codes for joining
cursor.execute("CREATE INDEX IF NOT EXISTS idx_municipal_entity ON municipal_boundaries(ENTITYNBR)")
print("Created index on ENTITYNBR")

# Index on normalized names for alternative matching
cursor.execute("CREATE INDEX IF NOT EXISTS idx_municipal_name ON municipal_boundaries(normalized_name)")
print("Created index on normalized_name")

# Index on county number for joining with county mapping
cursor.execute("CREATE INDEX IF NOT EXISTS idx_municipal_county ON municipal_boundaries(COUNTYNBR)")
print("Created index on COUNTYNBR")

# Index on tax entity codes (Excel uses EntityCode)
cursor.execute("CREATE INDEX IF NOT EXISTS idx_tax_entity ON tax_rates(EntityCode)")
print("Created index on tax_rates.EntityCode")

# Index on tax county names (Excel uses County)
cursor.execute("CREATE INDEX IF NOT EXISTS idx_tax_county ON tax_rates(County)")
print("Created index on tax_rates.County")

# Index on tax areas
cursor.execute("CREATE INDEX IF NOT EXISTS idx_tax_area ON tax_rates(TaxArea)")
print("Created index on tax_rates.TaxArea")

conn.commit()
conn.close()
print("\nAll indexes created successfully")

Adding indexes for performance...
Created index on ENTITYNBR
Created index on normalized_name
Created index on COUNTYNBR
Created index on tax_rates.EntityCode
Created index on tax_rates.County
Created index on tax_rates.TaxArea

All indexes created successfully


In [36]:
# Analyze multi-county cities
conn = sqlite3.connect(db_path)

print("Analyzing cities that span multiple counties...")
query = """
SELECT 
    NAME,
    COUNTYNBR,
    County as county_name,
    area_sq_miles,
    population,
    ROUND(area_sq_miles * 100.0 / SUM(area_sq_miles) OVER (PARTITION BY NAME), 2) as pct_of_total
FROM municipal_boundaries m
JOIN tax_rates t ON m.COUNTYNBR = t.CodeCounty
WHERE NAME IN (
    SELECT NAME 
    FROM municipal_boundaries 
    GROUP BY NAME 
    HAVING COUNT(*) > 1
)
GROUP BY NAME, COUNTYNBR, County, area_sq_miles, population
ORDER BY NAME, area_sq_miles DESC
"""
multi_county_cities = pd.read_sql_query(query, conn)
print("Multi-county cities:")
display(multi_county_cities)

# Summary statistics
print("\nSummary of multi-county cities:")
for city in multi_county_cities['NAME'].unique():
    city_data = multi_county_cities[multi_county_cities['NAME'] == city]
    total_area = city_data['area_sq_miles'].sum()
    total_pop = city_data['population'].iloc[0]  # Same across entries
    print(f"\n{city}:")
    print(f"  Total area: {total_area:.2f} sq mi")
    print(f"  Population: {int(total_pop):,}")
    for _, row in city_data.iterrows():
        print(f"  {row['county_name']}: {row['area_sq_miles']:.2f} sq mi ({row['pct_of_total']}%)")

conn.close()
print("\nNote: Areas represent county-specific portions, not total city area.")

Analyzing cities that span multiple counties...
Multi-county cities:


,NAME,COUNTYNBR,county_name,area_sq_miles,population,pct_of_total
0,Bluffdale,18,SALT LAKE,18.035073,19080.0,93.90
1,Bluffdale,25,UTAH,1.171328,19080.0,6.10
2,Draper,18,SALT LAKE,38.047134,50731.0,77.42
3,Draper,25,UTAH,11.097566,50731.0,22.58
4,Park City,22,SUMMIT,37.994962,8374.0,98.04
5,Park City,26,WASATCH,0.758740,8374.0,1.96
6,Santaquin,25,UTAH,17.706123,16898.0,98.11
7,Santaquin,12,JUAB,0.340731,16898.0,1.89



Summary of multi-county cities:

Bluffdale:
  Total area: 19.21 sq mi
  Population: 19,080
  SALT LAKE: 18.04 sq mi (93.9%)
  UTAH: 1.17 sq mi (6.1%)

Draper:
  Total area: 49.14 sq mi
  Population: 50,731
  SALT LAKE: 38.05 sq mi (77.42%)
  UTAH: 11.10 sq mi (22.58%)

Park City:
  Total area: 38.75 sq mi
  Population: 8,374
  SUMMIT: 37.99 sq mi (98.04%)
  WASATCH: 0.76 sq mi (1.96%)

Santaquin:
  Total area: 18.05 sq mi
  Population: 16,898
  UTAH: 17.71 sq mi (98.11%)
  JUAB: 0.34 sq mi (1.89%)

Note: Areas represent county-specific portions, not total city area.


In [37]:
# Analyze entity code structure
conn = sqlite3.connect(db_path)

print("Analyzing entity code structure by first digit...")
query = """
SELECT 
    CAST(EntityCode AS INTEGER) / 1000 as first_digit,
    COUNT(*) as record_count,
    COUNT(DISTINCT EntityName) as unique_entities
FROM tax_rates 
GROUP BY first_digit 
ORDER BY first_digit
"""
entity_analysis = pd.read_sql_query(query, conn)
print("Entity code distribution by first digit:")
display(entity_analysis)

print("\nEntity code meanings (from Utah State Tax Commission):")
print("1xxx = Counties")
print("2xxx = School districts") 
print("3xxx = Incorporated cities/towns")
print("4xxx = Special service districts")
print("6xxx = Special districts")
print("8xxx/9xxx = Redevelopment or community development areas")

print("\nSample entities by first digit:")
for digit in [1, 2, 3, 4, 6]:
    query = f"""
    SELECT 
        EntityName,
        EntityCode,
        COUNT(*) as frequency
    FROM tax_rates 
    WHERE CAST(EntityCode AS INTEGER) / 1000 = {digit}
    GROUP BY EntityName, EntityCode
    ORDER BY frequency DESC
    LIMIT 5
    """
    result = pd.read_sql_query(query, conn)
    print(f"\nFirst digit {digit}:")
    display(result)

conn.close()

Analyzing entity code structure by first digit...
Entity code distribution by first digit:


,first_digit,record_count,unique_entities
0,1,5181,32
1,2,1727,42
2,3,1250,256
3,4,5747,218
4,5,62,45
5,6,1089,13



Entity code meanings (from Utah State Tax Commission):
1xxx = Counties
2xxx = School districts
3xxx = Incorporated cities/towns
4xxx = Special service districts
6xxx = Special districts
8xxx/9xxx = Redevelopment or community development areas

Sample entities by first digit:

First digit 1:


,EntityName,EntityCode,frequency
0,MULTICOUNTY ASSESSING & COLLECTING LEVY,1015,1727
1,COUNTY ASSESSING & COLLECTING LEVY,1020,1717
2,SALT LAKE,1010,406
3,WEBER,1010,282
4,UTAH,1010,152



First digit 2:


...

,EntityName,EntityCode,frequency
0,SANDY CITY,3080,56
1,SALT LAKE CITY,3070,40
2,SALT LAKE LIBRARY,3071,40
3,WEST VALLEY CITY,3120,36
4,WEST JORDAN CITY,3110,33



First digit 4:


,EntityName,EntityCode,frequency
0,WEBER BASIN WATER CONSERVANCY DISTRICT,4005,467
1,CENTRAL UTAH WATER CONSERVANCY DISTRICT,4185,406
2,SOUTH SALT LAKE VALLEY MOSQUITO ABATEMENT DIST...,4040,321
3,WEBER AREA DISPATCH 911 AND EMERGENCY SERVICES...,4320,282
4,WEBER COUNTY MOSQUITO ABATEMENT DISTRICT,4080,282



First digit 6:


,EntityName,EntityCode,frequency
0,SALT LAKE COUNTY LIBRARY,6030,339
1,WEBER FIRE DISTRICT - BOND ( est. 1/1/06),6080,179
2,ANIMAL WELFARE SERVICES DISTRICT (COUNTY WIDE),6000,128
3,COUNTY LIBRARY,6030,128
4,MUNICIPAL TYPE SERVICES,6090,76


In [42]:
# hey devin
import sqlite3
import pandas as pd
from pathlib import Path

# City-level tax burden comparison (Approach 2)
#
# Goal: Compare the total property tax burden attributable to city-level decisions,
# to enable apples-to-apples comparison between cities.
#
# The problem with using just the 3xxx (city) rate:
#   Some cities provide fire, police, parks, etc. directly (funded in 3xxx).
#   Others outsource those services to special districts (4xxx/6xxx).
#   So 3xxx alone drastically understates the burden for cities that use districts.
#   e.g., West Haven CITY rate = 0.0 but pays 0.001083 to Weber Fire District (4xxx).
#
# Approach 2: total rate - 1xxx (county) - 2xxx (school)
#   County and school rates are set county-wide and identical for all cities in
#   a county. Subtracting them leaves only what varies city by city.
#   This captures: city general fund, fire, police, parks, water, sewer,
#   cemeteries, mosquito abatement, libraries, etc. — everything a city-level
#   decision shapes.
#
# IMPORTANT: The unique identifier for a tax area is (County, TaxArea, TaxAreaExtension).
# This version considers ALL tax areas for each city (including extensions)
# and reports min/max burden along with the corresponding TaxArea IDs
# formatted as "TaxArea - TaxAreaExtension".

conn = sqlite3.connect(db_path)

query = """
WITH city_tax_areas AS (
    -- Get all (County, TaxArea, TaxAreaExtension) combinations that have a city levy
    -- The unique identifier is County + TaxArea + TaxAreaExtension
    SELECT DISTINCT County, EntityName AS city_name, TaxArea, TaxAreaExtension
    FROM tax_rates
    WHERE CAST(EntityCode AS INTEGER) / 1000 = 3
),
breakdown AS (
    -- Calculate city burden for each unique (County, TaxArea, TaxAreaExtension)
    -- We must filter by County to ensure we're looking at the correct county's portion
    SELECT cta.County, cta.city_name, t.TaxArea, t.TaxAreaExtension,
        SUM(t.FinalRate) AS grand_total,
        SUM(CASE WHEN CAST(t.EntityCode AS INTEGER) / 1000 IN (1,2) THEN t.FinalRate ELSE 0 END) AS county_school,
        SUM(CASE WHEN CAST(t.EntityCode AS INTEGER) / 1000 NOT IN (1,2) THEN t.FinalRate ELSE 0 END) AS city_burden
    FROM tax_rates t
    JOIN city_tax_areas cta ON t.County = cta.County AND t.TaxArea = cta.TaxArea AND t.TaxAreaExtension = cta.TaxAreaExtension
    GROUP BY cta.County, cta.city_name, t.TaxArea, t.TaxAreaExtension
),
ranked_areas AS (
    SELECT 
        County, city_name, TaxArea, TaxAreaExtension, city_burden,
        ROW_NUMBER() OVER (PARTITION BY County, city_name ORDER BY city_burden ASC, TaxAreaExtension ASC) as min_rank,
        ROW_NUMBER() OVER (PARTITION BY County, city_name ORDER BY city_burden DESC, TaxAreaExtension DESC) as max_rank
    FROM breakdown
),
min_max_areas AS (
    SELECT 
        County, city_name,
        MIN(city_burden) AS min_city_burden,
        MAX(city_burden) AS max_city_burden,
        MAX(CASE WHEN min_rank = 1 THEN TaxArea END) AS min_tax_area,
        MAX(CASE WHEN min_rank = 1 THEN TaxAreaExtension END) AS min_tax_area_ext,
        MAX(CASE WHEN max_rank = 1 THEN TaxArea END) AS max_tax_area,
        MAX(CASE WHEN max_rank = 1 THEN TaxAreaExtension END) AS max_tax_area_ext
    FROM ranked_areas
    GROUP BY County, city_name
)
SELECT County, city_name, min_city_burden, max_city_burden, min_tax_area, min_tax_area_ext, max_tax_area, max_tax_area_ext
FROM min_max_areas
ORDER BY County, max_city_burden DESC
"""

city_burden_df = pd.read_sql_query(query, conn)

# Create reports folder and write report to markdown file
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
report_path = reports_dir / "city_tax_burden_report.md"

with open(report_path, 'w') as f:
    f.write("# City-Level Tax Burden Report (Approach 2)\n\n")
    f.write("**Methodology:** Total tax rate minus county (1xxx) and school (2xxx) rates.\n")
    f.write("**Reporting:** Min and max burden across ALL tax areas for each city (including extensions).\n")
    f.write("**Note:** Tax areas are uniquely identified by (County, TaxArea, TaxAreaExtension).\n\n")
    f.write("## Column Definitions\n\n")
    f.write("- **Min Local-ish Rate**: Minimum city-level burden across all tax areas\n")
    f.write("- **Max Local-ish Rate**: Maximum city-level burden across all tax areas\n")
    f.write("- **Min Tax Area**: TaxArea ID for the minimum burden (format: TaxArea - TaxAreaExtension)\n")
    f.write("- **Max Tax Area**: TaxArea ID for the maximum burden (format: TaxArea - TaxAreaExtension)\n\n")
    f.write("---\n\n")
    
    for county in city_burden_df['County'].unique():
        county_df = city_burden_df[city_burden_df['County'] == county].reset_index(drop=True)
        f.write(f"## {county}\n\n")
        f.write("| City | Min Local-ish Rate | Max Local-ish Rate | Min Tax Area | Max Tax Area |\n")
        f.write("|------|-------------------|-------------------|--------------|--------------|\n")
        for _, row in county_df.iterrows():
            min_area = f"{row['min_tax_area']} - {int(row['min_tax_area_ext']):04d}"
            max_area = f"{row['max_tax_area']} - {int(row['max_tax_area_ext']):04d}"
            f.write(f"| {row['city_name']} | {row['min_city_burden']:.6f} | {row['max_city_burden']:.6f} | {min_area} | {max_area} |\n")
        f.write("\n")

print(f"Report written to {report_path}")

conn.close()

Report written to reports/city_tax_burden_report.md
